<a href="https://colab.research.google.com/github/kyle29-git/kyle29-git_64061/blob/main/Assignment_2_Convolution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Import libraries
import os
import zipfile
import shutil
import math
import random
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

In [6]:
# Upload data
from google.colab import files

files.upload()

Saving train.zip to train.zip


In [4]:
# Define the local directory paths and create train, validation, test split

zip_path = '/content/train.zip'
extract_dir = '/content/train_unzipped'

# Unzip
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

base_dir = extract_dir
train_folder = os.path.join(base_dir, 'train')

cats_dir = os.path.join(base_dir, 'cats')
dogs_dir = os.path.join(base_dir, 'dogs')

os.makedirs(cats_dir, exist_ok=True)
os.makedirs(dogs_dir, exist_ok=True)

# Move images from train into cats and dogs
for filename in os.listdir(train_folder):
    file_path = os.path.join(train_folder, filename)

    if filename.startswith('cat'):
        shutil.move(file_path, os.path.join(cats_dir, filename))
    elif filename.startswith('dog'):
        shutil.move(file_path, os.path.join(dogs_dir, filename))

# Remove now-empty train folder
shutil.rmtree(train_folder)


In [6]:
# Ceate splits and batch size
full_ds = tf.keras.preprocessing.image_dataset_from_directory(
    base_dir,
    image_size=(150,150),
    batch_size=1,      # IMPORTANT
    shuffle=True,
    seed=123
)

print("Classes:", full_ds.class_names)

train_size = 1000
val_size = 500
test_size = 500

train_ds = full_ds.take(train_size)
remaining = full_ds.skip(train_size)

val_ds = remaining.take(val_size)
test_ds = remaining.skip(val_size).take(test_size)

BATCH_SIZE = 32

train_ds = train_ds.unbatch().batch(BATCH_SIZE)
val_ds = val_ds.unbatch().batch(BATCH_SIZE)
test_ds = test_ds.unbatch().batch(BATCH_SIZE)




Found 25000 files belonging to 2 classes.
Classes: ['cats', 'dogs']


In [7]:
# Normalize the images
normalization_layer = layers.Rescaling(1./255)
full_ds = full_ds.map(lambda x, y: (normalization_layer(x), y))

In [8]:
# Build model

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

model = tf.keras.Sequential([
    tf.keras.Input(shape=(150,150,3)),

    data_augmentation,
    layers.Rescaling(1./255),

    layers.Conv2D(32, 3, activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(64, 3, activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(128, 3, activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(256, 3, activation='relu'),
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential (Sequential)         │ (None, 150, 150, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling_2 (Rescaling)         │ (None, 150, 150, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 148, 148, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 74, 74, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 72, 72, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 36, 36, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 34, 34, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 17, 17, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 15, 15, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 7, 7, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 12544)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     1,605,760 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,994,305 (7.61 MB)

 Trainable params: 1,994,305 (7.61 MB)

 Non-trainable params: 0 (0.00 B)

In [111]:
# Part 1: train from scratch using 1000 images for training sample
history_1000 = model.fit(
    train_ds,
    epochs=20,
    steps_per_epoch=1000 // 32,
    validation_data=val_ds,
    validation_steps=500 // 32
)

test_loss_1000, test_acc_1000 = model.evaluate(test_ds)
print("Test accuracy (1000 training):", test_acc_1000)



Epoch 1/20
31/31 ━━━━━━━━━━━━━━━━━━━━ 74s 2s/step - accuracy: 0.5001 - loss: 0.6945 - val_accuracy: 0.5063 - val_loss: 0.6883
Epoch 2/20
31/31 ━━━━━━━━━━━━━━━━━━━━ 69s 2s/step - accuracy: 0.5276 - loss: 0.6887 - val_accuracy: 0.5104 - val_loss: 0.6850
Epoch 3/20
31/31 ━━━━━━━━━━━━━━━━━━━━ 80s 3s/step - accuracy: 0.5030 - loss: 0.6890 - val_accuracy: 0.5104 - val_loss: 0.6856
Epoch 4/20
 1/31 ━━━━━━━━━━━━━━━━━━━━ 1:01 2s/step - accuracy: 0.5417 - loss: 0.6670

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:160: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


31/31 ━━━━━━━━━━━━━━━━━━━━ 12s 329ms/step - accuracy: 0.5417 - loss: 0.6670 - val_accuracy: 0.5292 - val_loss: 0.6817
Epoch 5/20
31/31 ━━━━━━━━━━━━━━━━━━━━ 69s 2s/step - accuracy: 0.5777 - loss: 0.6786 - val_accuracy: 0.5958 - val_loss: 0.6569
Epoch 6/20
31/31 ━━━━━━━━━━━━━━━━━━━━ 69s 2s/step - accuracy: 0.6234 - loss: 0.6502 - val_accuracy: 0.6083 - val_loss: 0.6499
Epoch 7/20
31/31 ━━━━━━━━━━━━━━━━━━━━ 82s 3s/step - accuracy: 0.6268 - loss: 0.6534 - val_accuracy: 0.6271 - val_loss: 0.6448
Epoch 8/20
31/31 ━━━━━━━━━━━━━━━━━━━━ 12s 369ms/step - accuracy: 0.7917 - loss: 0.5484 - val_accuracy: 0.6458 - val_loss: 0.6393
Epoch 9/20
31/31 ━━━━━━━━━━━━━━━━━━━━ 82s 3s/step - accuracy: 0.6397 - loss: 0.6320 - val_accuracy: 0.6208 - val_loss: 0.6596
Epoch 10/20
31/31 ━━━━━━━━━━━━━━━━━━━━ 68s 2s/step - accuracy: 0.6638 - loss: 0.6104 - val_accuracy: 0.6646 - val_loss: 0.6112
Epoch 11/20
31/31 ━━━━━━━━━━━━━━━━━━━━ 81s 3s/step - accuracy: 0.6773 - loss: 0.6089 - val_accuracy: 0.6708 - val_loss: 0.

In [114]:
# Part 2: Train from scratch using 3000 images for training sample

history_3000 = model.fit(
    train_ds,
    epochs=20,
    steps_per_epoch=3000 // 32,
    validation_data=val_ds,
    validation_steps=500 // 32
)

test_loss_3000, test_acc_3000 = model.evaluate(test_ds)
print("Test accuracy (3000 training):", test_acc_3000)


Epoch 1/20
93/93 ━━━━━━━━━━━━━━━━━━━━ 187s 2s/step - accuracy: 0.5039 - loss: 0.6922 - val_accuracy: 0.5042 - val_loss: 0.6912
Epoch 2/20
93/93 ━━━━━━━━━━━━━━━━━━━━ 11s 104ms/step - accuracy: 0.5833 - loss: 0.6587 - val_accuracy: 0.5042 - val_loss: 0.6869
Epoch 3/20
93/93 ━━━━━━━━━━━━━━━━━━━━ 184s 2s/step - accuracy: 0.6039 - loss: 0.6686 - val_accuracy: 0.6479 - val_loss: 0.6399
Epoch 4/20
93/93 ━━━━━━━━━━━━━━━━━━━━ 12s 119ms/step - accuracy: 0.5833 - loss: 0.6059 - val_accuracy: 0.6458 - val_loss: 0.6431
Epoch 5/20
93/93 ━━━━━━━━━━━━━━━━━━━━ 201s 2s/step - accuracy: 0.6479 - loss: 0.6314 - val_accuracy: 0.6104 - val_loss: 0.6463
Epoch 6/20
93/93 ━━━━━━━━━━━━━━━━━━━━ 21s 216ms/step - accuracy: 0.6667 - loss: 0.5659 - val_accuracy: 0.6375 - val_loss: 0.6318
Epoch 7/20
93/93 ━━━━━━━━━━━━━━━━━━━━ 202s 2s/step - accuracy: 0.6700 - loss: 0.6085 - val_accuracy: 0.6562 - val_loss: 0.6149
Epoch 8/20
93/93 ━━━━━━━━━━━━━━━━━━━━ 12s 117ms/step - accuracy: 0.7500 - loss: 0.5198 - val_accuracy: 0.

In [117]:
# Part 3: Train from scratch using optimal amount of images for training sample
history_500 = model.fit(
    train_ds,
    epochs=20,
    steps_per_epoch=500 // 32,
    validation_data=val_ds,
    validation_steps=500 // 32
)

test_loss_500, test_acc_500 = model.evaluate(test_ds)
print("Test accuracy (500 training):", test_acc_500)

Epoch 1/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 42s 3s/step - accuracy: 0.4958 - loss: 0.6955 - val_accuracy: 0.5396 - val_loss: 0.6915
Epoch 2/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 10s 643ms/step - accuracy: 0.3500 - loss: 0.7003 - val_accuracy: 0.5417 - val_loss: 0.6919
Epoch 3/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 39s 3s/step - accuracy: 0.5592 - loss: 0.6897 - val_accuracy: 0.5521 - val_loss: 0.6896
Epoch 4/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 9s 592ms/step - accuracy: 0.4000 - loss: 0.7012 - val_accuracy: 0.5875 - val_loss: 0.6895
Epoch 5/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 38s 3s/step - accuracy: 0.6176 - loss: 0.6857 - val_accuracy: 0.5250 - val_loss: 0.6873
Epoch 6/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 13s 843ms/step - accuracy: 0.4000 - loss: 0.7034 - val_accuracy: 0.5396 - val_loss: 0.6866
Epoch 7/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 42s 3s/step - accuracy: 0.6322 - loss: 0.6813 - val_accuracy: 0.6042 - val_loss: 0.6826
Epoch 8/20
15/15 ━━━━━━━━━━━━━━━━━━━━ 9s 545ms/step - accuracy: 0.4500 - loss: 0.6958 - val_accuracy: 0.5854 -

In [120]:
history_5000 = model.fit(
    train_ds,
    epochs=20,
    steps_per_epoch=5000 // 32,
    validation_data=val_ds,
    validation_steps=500 // 32
)

test_loss_5000, test_acc_5000 = model.evaluate(test_ds)
print("Test accuracy (5000 training):", test_acc_5000)

Epoch 1/20
156/156 ━━━━━━━━━━━━━━━━━━━━ 307s 2s/step - accuracy: 0.5438 - loss: 0.6882 - val_accuracy: 0.5354 - val_loss: 0.7118
Epoch 2/20
  1/156 ━━━━━━━━━━━━━━━━━━━━ 1:01 398ms/step - accuracy: 0.5000 - loss: 0.7516

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:160: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


156/156 ━━━━━━━━━━━━━━━━━━━━ 13s 81ms/step - accuracy: 0.5000 - loss: 0.7516 - val_accuracy: 0.5208 - val_loss: 0.7735
Epoch 3/20
156/156 ━━━━━━━━━━━━━━━━━━━━ 307s 2s/step - accuracy: 0.6173 - loss: 0.6486 - val_accuracy: 0.6396 - val_loss: 0.6451
Epoch 4/20
156/156 ━━━━━━━━━━━━━━━━━━━━ 21s 130ms/step - accuracy: 0.6250 - loss: 0.7135 - val_accuracy: 0.6167 - val_loss: 0.7048
Epoch 5/20
156/156 ━━━━━━━━━━━━━━━━━━━━ 322s 2s/step - accuracy: 0.6789 - loss: 0.6044 - val_accuracy: 0.6542 - val_loss: 0.6205
Epoch 6/20
156/156 ━━━━━━━━━━━━━━━━━━━━ 13s 80ms/step - accuracy: 0.5000 - loss: 0.7669 - val_accuracy: 0.6333 - val_loss: 0.6801
Epoch 7/20
156/156 ━━━━━━━━━━━━━━━━━━━━ 304s 2s/step - accuracy: 0.6987 - loss: 0.5735 - val_accuracy: 0.7208 - val_loss: 0.5527
Epoch 8/20
156/156 ━━━━━━━━━━━━━━━━━━━━ 21s 130ms/step - accuracy: 0.7500 - loss: 0.5692 - val_accuracy: 0.6500 - val_loss: 0.6499
Epoch 9/20
156/156 ━━━━━━━━━━━━━━━━━━━━ 322s 2s/step - accuracy: 0.7030 - loss: 0.5719 - val_accuracy:

In [9]:
history_10000 = model.fit(
    train_ds,
    epochs=20,
    steps_per_epoch=10000 // 32,
    validation_data=val_ds,
    validation_steps=500 // 32
)

test_loss_10000, test_acc_10000 = model.evaluate(test_ds)
print("Test accuracy (10000 training):", test_acc_10000)

Epoch 1/20
312/312 ━━━━━━━━━━━━━━━━━━━━ 610s 2s/step - accuracy: 0.5366 - loss: 0.6818 - val_accuracy: 0.6438 - val_loss: 0.6282
Epoch 2/20
  1/312 ━━━━━━━━━━━━━━━━━━━━ 4:19 835ms/step - accuracy: 0.5000 - loss: 0.6504

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:160: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


312/312 ━━━━━━━━━━━━━━━━━━━━ 19s 59ms/step - accuracy: 0.5000 - loss: 0.6504 - val_accuracy: 0.6375 - val_loss: 0.6309
Epoch 3/20
312/312 ━━━━━━━━━━━━━━━━━━━━ 603s 2s/step - accuracy: 0.6758 - loss: 0.6078 - val_accuracy: 0.7125 - val_loss: 0.5549
Epoch 4/20
312/312 ━━━━━━━━━━━━━━━━━━━━ 19s 60ms/step - accuracy: 0.7500 - loss: 0.6514 - val_accuracy: 0.7125 - val_loss: 0.5484
Epoch 5/20
312/312 ━━━━━━━━━━━━━━━━━━━━ 622s 2s/step - accuracy: 0.7118 - loss: 0.5574 - val_accuracy: 0.7479 - val_loss: 0.5135
Epoch 6/20
312/312 ━━━━━━━━━━━━━━━━━━━━ 22s 67ms/step - accuracy: 0.6250 - loss: 0.6708 - val_accuracy: 0.7500 - val_loss: 0.5080
Epoch 7/20
312/312 ━━━━━━━━━━━━━━━━━━━━ 598s 2s/step - accuracy: 0.7366 - loss: 0.5202 - val_accuracy: 0.7500 - val_loss: 0.5070
Epoch 8/20
312/312 ━━━━━━━━━━━━━━━━━━━━ 19s 60ms/step - accuracy: 0.5000 - loss: 0.7301 - val_accuracy: 0.7833 - val_loss: 0.4756
Epoch 9/20
312/312 ━━━━━━━━━━━━━━━━━━━━ 622s 2s/step - accuracy: 0.7522 - loss: 0.4971 - val_accuracy: 0

In [121]:
# Part 4: Pretrained network model
conv_base = keras.applications.VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(150,150,3)
)

conv_base.trainable = False

model_pre = keras.Sequential([
    conv_base,
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

model_pre.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)


58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [127]:
# Pretrained model with 1000 images
history_pre_1000 = model_pre.fit(
    train_ds,
    epochs=20,
    steps_per_epoch=math.ceil(1000 / 32),
    validation_data=val_ds,
    validation_steps=math.ceil(500 / 32),
    callbacks=[early_stop]
)
test_loss_pre_1000, test_acc_pre_1000 = model_pre.evaluate(test_ds)
print("Pretrained Test accuracy (1000):", test_acc_pre_1000)


Epoch 1/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 425s 13s/step - accuracy: 0.9308 - loss: 0.9508 - val_accuracy: 0.9160 - val_loss: 0.9531
Epoch 2/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 128s 4s/step - accuracy: 0.0000e+00 - loss: 0.0000e+00 - val_accuracy: 0.9140 - val_loss: 0.9609
Epoch 3/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 399s 13s/step - accuracy: 0.9718 - loss: 0.4146 - val_accuracy: 0.9200 - val_loss: 0.9897
Epoch 4/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 141s 4s/step - accuracy: 0.0000e+00 - loss: 0.0000e+00 - val_accuracy: 0.9200 - val_loss: 0.9897
16/16 ━━━━━━━━━━━━━━━━━━━━ 128s 8s/step - accuracy: 0.9176 - loss: 0.6464
Pretrained Test accuracy (1000): 0.9240000247955322
